# 🔬 Inspecting a DataFrame

**Time:** ~20 minutes &nbsp;•&nbsp; **Goal:** open the pollinator survey as it came off the clipboard, and work out both what is in it and what is wrong with it.

The lecture used a tidied extract — 13 neat columns, nothing to trip over. This is the real file: 22 columns, 1,620 watches, two summers, and a mistake in the header row.

Two of the things you find here look like errors and are not. Deciding that is the actual skill.

## Our data — nine allotments, two summers

Volunteers walked a plot, picked a flowering crop, watched it for a fixed time, and wrote down every insect that landed. **One row = one watch.**

| column | what it is |
|---|---|
| `Year`, `Date`, `Month` | when |
| `Allotment`, `Plot`, `Community` | where — nine sites, 83 plots, shared or individual |
| `Crop_Generic` | which crop was being watched |
| `Plants`, `Total_Flowers` | how many plants and open flowers were in view |
| `Beetles` … `Wasps` | visit counts, one column per insect group |
| `Terrestris` … `Hortorum` | six **bumblebee species**, summing to `Bumblebees` |

In [ ]:
# Run this first. It fetches the survey file and opens it.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]

print("loaded", SURVEY)
print(df.shape)

## Skill 1 — A DataFrame will describe itself

You have 1,620 rows and you are never going to read them. So the first thing you do with any new table is ask it four questions:

```python
df.shape      # (rows, columns)
df.columns    # the names
df.dtypes     # what kind of data is in each
df.head()     # what the rows actually look like
```

`df.info()` rolls three of those into one and adds the thing you most need: **how many values are actually there**, column by column.

In [ ]:
# DEMO - four questions, asked of a table you have never seen
print("rows and columns:", df.shape)
print()
print(df.columns.tolist())
print()
df.head()

In [ ]:
# DEMO - info() is the one to reach for. Read the 'Non-Null Count' column.
df.info()

### 🐝 DIY challenge

1. Print the **last** three rows with `df.tail(3)`. Which year are they from?
2. `df.shape` is a tuple. Print just the number of **columns**, using an index.
3. Run `df.describe()`. It summarises every *numeric* column. Which columns does it leave out, and why?

> 💡 `.head()` and `.tail()` take a number: `df.head(10)`. `.describe()` needs no argument.

In [ ]:
# TODO - your turn

## Skill 2 — The column that isn't there

Look at `df.info()` again. One of the bumblebee species columns is not called what you think it is called.

This is the commonest way a real analysis fails on day one: somebody typed a space after a column name in Excel three years ago, and now `df["Hortorum"]` raises a `KeyError` while `df["Hortorum "]` works — and the two look identical on screen.

The fix is one line, and it belongs in every script you start:

```python
df.columns = df.columns.str.strip()
```

`.str` applies a string method to **every name at once**. You will meet it again on data columns.

In [ ]:
# DEMO - the bug, made visible
try:
    df["Hortorum"]
except KeyError as e:
    print("KeyError:", e)

# repr() shows the quotes, so you can finally see the space
print()
for name in df.columns:
    if name != name.strip():
        print("suspicious:", repr(name))

In [ ]:
# DEMO - and the fix
df.columns = df.columns.str.strip()

print(df["Hortorum"].sum(), "Bombus hortorum visits recorded")

### 🐝 DIY challenge

1. Before the fix, how many column names had stray whitespace? Write a line that **counts** them rather than printing them.
2. Column names can be messy in other ways too. Print `df.columns.str.lower()` — what would that be useful for?
3. `Crop_Generic` is a clumsy name. Rename it to `Crop` with
   `df = df.rename(columns={"Crop_Generic": "Crop"})`, then check `df.columns`.

> 💡 A comparison over a whole collection gives you `True`/`False` per item, and `sum()` counts the `True`s — exactly as it did for numpy masks.

In [ ]:
# TODO - your turn

## Skill 3 — A column is a Series

This is the idea that makes pandas click:

> A **DataFrame** is a collection of **Series**, side by side, sharing one index.

So `df["Honeybees"]` is a Series — a single labelled column — and almost every method you can call on one, you can also call on the whole DataFrame, in which case you get **one answer per column**.

```python
df["Honeybees"].sum()             # one number
df[["Honeybees", "Bumblebees"]]   # a DataFrame - note the double brackets
df[INSECTS].sum()                 # one total per insect column
```

The double bracket catches everybody. `df["a"]` asks for *one* column and gives a Series; `df[["a"]]` hands pandas a **list** of names and gives a DataFrame back.

That last line answers a real question: **which pollinator group is doing the work on Brighton allotments?**

In [ ]:
# DEMO - one column, and several
bees = df["Honeybees"]

print(type(bees))
print("total honeybee visits:", bees.sum())
print("busiest single watch: ", bees.max())
print("mean per watch:       ", round(bees.mean(), 3))

print()
pair = df[["Honeybees", "Bumblebees"]]
print(type(pair))
pair.head()

In [ ]:
# DEMO - the headline result of the whole survey, in one line
totals = df[INSECTS].sum()
print(totals.sort_values(ascending=False))


### 🔬 DIY challenge

1. How many **bumblebee** visits were recorded in total across the whole survey?
2. `df["Allotment"].value_counts()` — which allotment was surveyed most? `.value_counts()` is the most useful method in pandas for a text column; use it on everything.
3. `totals` is itself a Series. What is its `.idxmax()`, and how does that differ from `.max()`?
4. How many watches saw **no insects at all**? Build the row total with `df[INSECTS].sum(axis=1)`.

> 💡 `axis=0` goes down the columns (the default), `axis=1` goes across the rows. Getting this the wrong way round is a rite of passage.

In [ ]:
# TODO - your turn

## Skill 4 — Where are the holes?

`.isna().sum()` gives you a count of missing values per column, and it should be the second thing you ever run on a new file.

`NaN` is contagious: any arithmetic touching it gives `NaN` back. That is deliberate — it stops a missing value silently becoming a zero.

But the count is not the interesting part. **Look at which rows they are.**

In [ ]:
# DEMO - only one column has holes in it
missing = df.isna().sum()
print(missing[missing > 0])

print()
print("rows affected:", df["Plants"].isna().sum(), "of", len(df))

In [ ]:
# DEMO - who is missing, and does it look random?
gaps = df[df["Plants"].isna()]

print(gaps["Crop_Generic"].value_counts())
print()
print(gaps["Year"].value_counts())

Look at that first table before you touch anything. Two thirds of the missing plant counts are **blackberry** — and blackberry on an allotment is a bramble along a fence, not a countable number of plants.

The gap is not random noise. It is a category of crop where the question did not really apply, and that changes what you should do about it.

So what do you do about it? Three options, three different claims about the world:

| what you write | what you are claiming |
|---|---|
| `df.dropna(subset=["Plants"])` | "These watches are unusable for my question." |
| `df["Plants"].fillna(0)` | "There were no plants." — here that is simply a lie. |
| `df["Plants"].fillna(df["Plants"].median())` | "A typical plot, near enough." — but you have invented 41 numbers. |
| leave it | "I am not using this column, so it cannot hurt me." |

For this survey the honest answer is the last one. We are counting **insects per flower**, and `Total_Flowers` has no gaps at all. Deleting 41 real watches to tidy a column you were not going to use is a bad trade.

In [ ]:
# DEMO - what each choice costs you
print("rows if we drop:", len(df.dropna(subset=["Plants"])), "of", len(df))

filled = df["Plants"].fillna(df["Plants"].median())
print()
print("median fill changes the mean from",
      round(df["Plants"].mean(), 2), "to", round(filled.mean(), 2))
print("...and the standard deviation from",
      round(df["Plants"].std(), 2), "to", round(filled.std(), 2))

The standard deviation moved **down** — 19.34 to 19.12. Only slightly, because 41 invented rows cannot do much damage among 1,620. But that is the direction you get every time: 41 perfectly average plots that nobody ever visited make the survey look more consistent than it was. On a file where a third of the values are missing, the same move is not slight at all.

### 🔬 DIY challenge

1. Compare `df["Plants"].count()` with `len(df)`. Why do they differ?
2. Build `df["Total_Flowers"] / df["Plants"]`. How many `NaN`s does it have, and why exactly that many?
3. Make a copy with `clean = df.copy()` and drop the missing rows from the copy. Now try `clean = df` instead and change something — what happens to `df`?

> 💡 `clean = df` does not make a new table. It makes a second name for the same one.

In [ ]:
# TODO - your turn

## Skill 5 — The repeats that aren't errors 🏆

`.duplicated()` marks every row that has appeared before, exactly, in every column.

```python
df.duplicated().sum()        # how many repeats
df[df.duplicated(keep=False)]  # show every member of every repeated group
```

Run it on this file and you get 33. The temptation is enormous: `df.drop_duplicates()`, one line, table is clean, move on.

Do not. Look at them first.

In [ ]:
# DEMO - look before you delete
print("rows flagged as repeats:", df.duplicated().sum())

repeats = df[df.duplicated(keep=False)]
print("rows involved in a repeated group:", len(repeats))

cols = ["Year", "Date", "Allotment", "Plot", "Crop_Generic", "Total_Flowers"] + INSECTS
repeats[cols].head(8)

In [ ]:
# DEMO - what do the repeats have in common?
print(repeats["Allotment"].value_counts())
print()
all_zero = repeats[INSECTS].sum(axis=1) == 0
print("repeats that recorded nothing at all:", all_zero.sum(), "of", len(repeats))

There it is. Within each repeated group it is the same plot, the same day, the same crop — and **53 of the 59 rows recorded nothing at all**.

A volunteer watched four separate courgette plants one afternoon. Nothing landed on any of them. Four rows, identical in every column, because there was nothing to tell them apart.

`drop_duplicates()` would throw three away — and every "visits per watch" number you calculated afterwards would be too high, because you deleted the empty watches and kept the busy ones.

**The rule:** a duplicate is only an error if you can say *how* it got there.

### 🏆 DIY finale — write down what you found

You have met three things wrong, or apparently wrong, with this file. Write **one sentence each**, of the kind you would put in a methods section:

1. The header bug — what was wrong, and what you did.
2. The 41 missing plant counts — drop, fill or leave? Why?
3. The 33 duplicate rows — errors or not? What is your evidence?

Then, in code: work out how many rows `df.drop_duplicates()` would remove, and what that does to the mean `Bumblebees`. Does the direction match the argument above?

> 💡 The sentences are the harder half, and in an assessed report they are worth more than the code.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- The pandas **10 minutes** tour: <https://pandas.pydata.org/docs/user_guide/10min.html>
- Working with missing data, in full: <https://pandas.pydata.org/docs/user_guide/missing_data.html>
- **Extra notebook:** `extras/01_cleaning_deeper` — renaming columns and translating values, real dates and a date index, and writing a clean file out.

## Recap

You can now:

- size up a new CSV with `.shape`, `.dtypes`, `.info()` and `.describe()`
- fix a whitespace bug in a header with `df.columns.str.strip()`
- tell a **Series** from a **DataFrame**, and remember why `df[["a", "b"]]` needs two brackets
- find missing values with `.isna().sum()` — and **look at which rows they are** before deciding
- use `.duplicated()` to find repeats, and **not** delete them reflexively

**Next:** `02_selecting_rows_and_columns` — asking the table for exactly the rows you want.